# Pose2Text —


In [ ]:
import os, sys, torch, shutil
print('Python:', sys.version)
print('PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU — select a GPU runtime')
print(shutil.disk_usage('/content'))


Python: 3.13.16 (main, Oct  1 2026, 15:40:24) [GCC 13.3.0]
PyTorch: 2.11.0+cu130
GPU: Tesla T4
usage(total=120942624768, used=45627932672, free=75297914880)


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
PERSIST_ROOT = Path('/content/drive/MyDrive/Pose2Text')
PERSIST_ROOT.mkdir(parents=True, exist_ok=True)
print('Persistent Drive folder:', PERSIST_ROOT)


Mounted at /content/drive
Persistent Drive folder: /content/drive/MyDrive/Pose2Text


## Google Drive persistence



In [ ]:
from pathlib import Path
import zipfile, os, shutil
PROJECT_ZIP = Path('/content/Pose2Text.zip')
PROJECT = Path('/content/Pose2Text')
DRIVE_ZIP = PERSIST_ROOT / 'Pose2Text.zip'
if DRIVE_ZIP.exists():
    try:
        with zipfile.ZipFile(DRIVE_ZIP) as archive:
            valid_zip = 'Pose2Text/requirements.txt' in archive.namelist()
    except zipfile.BadZipFile:
        valid_zip = False
    if not valid_zip:
        print('Drive mein purani/invalid project ZIP mili; dobara updated ZIP upload karo.')
        DRIVE_ZIP.unlink()
if not DRIVE_ZIP.exists():
    if PROJECT_ZIP.exists():
        with zipfile.ZipFile(PROJECT_ZIP) as archive:
            if 'Pose2Text/requirements.txt' not in archive.namelist():
                raise FileNotFoundError('Colab Files mein purani ZIP hai. Updated Pose2Text.zip upload karo.')
        shutil.copy2(PROJECT_ZIP, DRIVE_ZIP)
    elif not (PROJECT / 'requirements.txt').exists():
        from google.colab import files
        print('Apne computer se Pose2Text.zip select karke upload karo.')
        uploaded = files.upload()
        choices = [Path('/content') / name for name in uploaded
                   if name.lower().endswith('.zip') and 'pose2text' in name.lower()]
        if not choices:
            raise FileNotFoundError('Pose2Text naam wali ZIP upload nahi hui.')
        with zipfile.ZipFile(choices[0]) as archive:
            if 'Pose2Text/requirements.txt' not in archive.namelist():
                raise FileNotFoundError('Uploaded ZIP purani/invalid hai. Updated Pose2Text.zip chahiye.')
        shutil.copy2(choices[0], DRIVE_ZIP)
if DRIVE_ZIP.exists():
    with zipfile.ZipFile(DRIVE_ZIP) as archive:
        archive.extractall('/content')
if not (PROJECT / 'requirements.txt').is_file():
    raise FileNotFoundError('ZIP mein Pose2Text/requirements.txt nahi mila. Sahi Pose2Text.zip upload karo.')
os.chdir(PROJECT)
print('Project ready:', Path.cwd(), '| Drive copy:', DRIVE_ZIP)


Project ready: /content/Pose2Text | Drive copy: /content/drive/MyDrive/Pose2Text/Pose2Text.zip


In [ ]:
%pip install -q -r requirements.txt


  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [ ]:
CHECKPOINT_DIR = PERSIST_ROOT / 'checkpoints'
RESULTS_DIR = PERSIST_ROOT / 'results'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
BEST_CHECKPOINT = CHECKPOINT_DIR / 'best.pt'
SAVED_METRICS = RESULTS_DIR / 'metrics.json'
HAVE_SAVED_RUN = BEST_CHECKPOINT.is_file() and SAVED_METRICS.is_file()
print('Saved trained model:', BEST_CHECKPOINT.exists())
print('Saved evaluation:', SAVED_METRICS.exists())
print('Will skip dataset download/training:', HAVE_SAVED_RUN)


Saved trained model: True
Saved evaluation: True
Will skip dataset download/training: True


## Dataset setup (first run only)
45 GB free space required.


In [ ]:
DATA_ROOT = Path('/content/pose2text_data/HumanML3D')
if HAVE_SAVED_RUN:
    print('Saved checkpoint + evaluation found in Drive; skipping dataset download/extraction.')
else:
    from pathlib import Path
    import shutil, subprocess
    old_bad_data = Path('/content/HumanML3D')
    if old_bad_data.exists() and old_bad_data.is_dir() and not old_bad_data.is_symlink():
        # Remove only the exact old failed extraction folder from the earlier workflow.
        if old_bad_data.resolve() == Path('/content/HumanML3D').resolve():
            shutil.rmtree(old_bad_data)
            print('Removed previous failed extraction:', old_bad_data)
    DATA_ARCHIVE = Path('/content/HumanML3D.zip')
    DATA_PARENT = Path('/content/pose2text_data')
    DATA_ROOT = DATA_PARENT / 'HumanML3D'
    DATA_URL = 'https://huggingface.co/AlayaLab/FloodDiffusionDownloads/resolve/main/HumanML3D.zip'
    free_gb = shutil.disk_usage('/content').free / (1024**3)
    print(f'Free Colab disk: {free_gb:.1f} GB')
    if free_gb < 45:
        raise RuntimeError('Disk space kam hai. Colab runtime ke Files panel se purani badi files/folders hatao, phir runtime restart karke dobara try karo.')
    if DATA_ARCHIVE.exists():
        integrity = subprocess.run(['unzip', '-tq', str(DATA_ARCHIVE)], check=False)
        if integrity.returncode != 0:
            print('Existing ZIP is incomplete/corrupt; removing it before a fresh download.')
            DATA_ARCHIVE.unlink()
    if not DATA_ARCHIVE.exists():
        subprocess.run(['wget', '-c', '--show-progress', DATA_URL, '-O', str(DATA_ARCHIVE)], check=True)
    subprocess.run(['unzip', '-tq', str(DATA_ARCHIVE)], check=True)
    DATA_PARENT.mkdir(parents=True, exist_ok=True)
    if not DATA_ROOT.exists():
        subprocess.run(['unzip', '-oq', str(DATA_ARCHIVE), '-d', str(DATA_PARENT)], check=True)
    if not DATA_ROOT.exists():
        candidates = list(DATA_PARENT.rglob('new_joint_vecs'))
        if candidates:
            DATA_ROOT = candidates[0].parent
    print('Dataset root:', DATA_ROOT)
    print('Dataset root entries:', [p.name for p in DATA_ROOT.iterdir()][:20])


Saved checkpoint + evaluation found in Drive; skipping dataset download/extraction.


## 4. Dataset integrity check



In [ ]:
if HAVE_SAVED_RUN:
    print('Saved run found; dataset integrity scan is not needed to view the saved demo/results.')
else:
    assert (DATA_ROOT / 'texts').is_dir(), f'texts folder nahi mila: {DATA_ROOT}'
    assert (DATA_ROOT / 'new_joint_vecs').is_dir(), f'new_joint_vecs folder nahi mila: {DATA_ROOT}'
    for split in ('train.txt', 'val.txt', 'test.txt'):
        assert (DATA_ROOT / split).is_file(), f'{split} nahi mila: {DATA_ROOT}'
    import subprocess, sys
    subprocess.run([sys.executable, 'scripts/check_dataset.py', '--data-root', str(DATA_ROOT)], check=True)


Saved run found; dataset integrity scan is not needed to view the saved demo/results.


## 5. Real data batch aur model forward pass check


In [ ]:
if HAVE_SAVED_RUN:
    print('Saved trained run found; skipping sample load and synthetic smoke run.')
else:
    from src.data.dataset import HumanML3D
    ds = HumanML3D(str(DATA_ROOT))
    train_ids = ds.split_ids('train')
    print('Training IDs:', len(train_ids))
    sample_id = train_ids[0]
    motion = ds.motion(sample_id)
    captions = ds.captions(sample_id)
    print('Sample:', sample_id, '| motion:', motion.shape, '| captions:', len(captions), '| first:', captions[0] if captions else 'none')
    assert motion.ndim == 2 and motion.shape[0] > 0


Saved trained run found; skipping sample load and synthetic smoke run.


In [ ]:
if not HAVE_SAVED_RUN:
    import subprocess, sys
    subprocess.run([sys.executable, 'scripts/train.py', '--config', 'configs/smoke_test.yaml', '--smoke-test'], check=True)
    subprocess.run([sys.executable, 'scripts/evaluate.py', '--checkpoint', 'checkpoints/smoke/best.pt', '--results-dir', 'results/smoke'], check=True)
else:
    print('Skipping synthetic smoke run because the saved real-data run is already available.')


Skipping synthetic smoke run because the saved real-data run is already available.


## 6.first real training config


In [ ]:
import yaml
from pathlib import Path
cfg_path = Path('configs/default.yaml')
cfg = yaml.safe_load(cfg_path.read_text())
cfg.update({
    'data_root': str(DATA_ROOT),
    'motion_dir': 'new_joint_vecs',
    'batch_size': 16,
    'epochs': 3,
    'train_limit': 2000,
    'num_workers': 2,
    'checkpoint_dir': str(CHECKPOINT_DIR),
    'results_dir': str(RESULTS_DIR),
})
cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print(cfg_path.read_text())


data_root: /content/pose2text_data/HumanML3D
motion_dir: new_joint_vecs
max_motion_length: 196
max_caption_length: 40
d_model: 256
nhead: 4
encoder_layers: 3
decoder_layers: 3
dim_feedforward: 1024
dropout: 0.1
batch_size: 16
epochs: 3
learning_rate: 0.0002
weight_decay: 0.0001
grad_clip: 1.0
patience: 8
seed: 17
num_workers: 2
train_limit: 2000
checkpoint_dir: /content/drive/MyDrive/Pose2Text/checkpoints
results_dir: /content/drive/MyDrive/Pose2Text/results



In [ ]:
import subprocess, sys
if HAVE_SAVED_RUN:
    print('Drive mein completed checkpoint + results mil gaye. Training skip ki—dobara train nahi hoga.')
elif BEST_CHECKPOINT.exists():
    print('Partial/previous checkpoint found. Training isi checkpoint se resume karega.')
    subprocess.run([sys.executable, 'scripts/train.py', '--config', 'configs/default.yaml', '--resume', str(BEST_CHECKPOINT)], check=True)
else:
    print('First training run. Checkpoint har best epoch par Google Drive mein save hoga.')
    subprocess.run([sys.executable, 'scripts/train.py', '--config', 'configs/default.yaml'], check=True)


Drive mein completed checkpoint + results mil gaye. Training skip ki—dobara train nahi hoga.


## Test evaluation (once)


In [ ]:
import subprocess, sys
if SAVED_METRICS.exists():
    print('Saved test evaluation already exists in Drive; skipping reevaluation.')
else:
    subprocess.run([sys.executable, 'scripts/evaluate.py', '--checkpoint', str(BEST_CHECKPOINT), '--data-root', str(DATA_ROOT), '--results-dir', str(RESULTS_DIR)], check=True)


Saved test evaluation already exists in Drive; skipping reevaluation.


In [ ]:
import json
import pandas as pd
for name in ('metrics.json', 'model_comparison.csv', 'validation_history.csv'):
    p = RESULTS_DIR / name
    print('\n###', p, '| exists:', p.exists())
    if p.exists() and p.suffix == '.json': print(json.loads(p.read_text()))
    elif p.exists() and p.suffix == '.csv': display(pd.read_csv(p))
predictions = RESULTS_DIR / 'predictions.csv'
if predictions.exists():
    print('Saved real test-set caption examples:')
    display(pd.read_csv(predictions).head(10))



### /content/drive/MyDrive/Pose2Text/results/metrics.json | exists: True
{'dataset': 'HumanML3D', 'split': 'test', 'examples': 4384, 'tokenization': 'lowercase alphanumeric regex; whitespace tokens for metrics', 'metrics': [{'model': 'transformer', 'bleu_1': 0.5151376900246861, 'bleu_2': 0.3668183714337591, 'bleu_3': 0.22959824047313346, 'bleu_4': 0.15226284958311218, 'rouge_l': 0.42738170550108046, 'meteor': 0.3472941108810674}, {'model': 'majority', 'bleu_1': 0.3079825298668147, 'bleu_2': 0.21005550940208248, 'bleu_3': 0.10864578454718947, 'bleu_4': 0.08410835192333356, 'rouge_l': 0.355145725826969, 'meteor': 0.6416112522015937}, {'model': 'nearest_motion', 'bleu_1': 0.45699325824517906, 'bleu_2': 0.27705235948698725, 'bleu_3': 0.16308819330465282, 'bleu_4': 0.1061058073864498, 'rouge_l': 0.38862226321230803, 'meteor': None}], 'unavailable': {'bert_score': 'Not run; optional package/model weights not installed or downloaded', 'cider': 'Not implemented (optional)'}, 'caveat': 'Automa

,model,bleu_1,bleu_2,bleu_3,bleu_4,rouge_l,meteor
0,transformer,0.515138,0.366818,0.229598,0.152263,0.427382,0.347294
1,majority,0.307983,0.210056,0.108646,0.084108,0.355146,0.641611
2,nearest_motion,0.456993,0.277052,0.163088,0.106106,0.388622,NaN



### /content/drive/MyDrive/Pose2Text/results/validation_history.csv | exists: True


,epoch,train_loss,validation_loss


Saved real test-set caption examples:


,motion_id,transformer,majority,nearest_motion,nearest_motion_id,references
0,004822,a person walks forward and then stops,a person slowly walked forward,the figure is continously walking while his ar...,M004055,a person is walking in place at a slow pace. |...
1,014457,a person is doing a circle,a person slowly walked forward,a person grabbed something and hit with it som...,013129,the person swings a golf club. || a person swi...
2,009613,a person is doing a circle,a person slowly walked forward,a man runs backwards while occasionally lookin...,001605,the man runs back wards || the person is runni...
3,008463,a person walks forward and then stops,a person slowly walked forward,the person is picking up and moving something.,M001289,"a man walks forward, then squats to pick somet..."
4,014160,a person raises their right hand to their righ...,a person slowly walked forward,a standing person raises their right arm and w...,006377,a person waves with his right hand. || a perso...
5,002530,a person walks forward and then stops,a person slowly walked forward,a man walks gently forwards a few steps.,004207,a person walks towards the camera. || a person...
6,004945,a person stands up and sits down and then stan...,a person slowly walked forward,bending down and grabbing something.,011546,"person is bent down trying to pick up stuff, l..."
7,001969,a person walks forward turns around and walks ...,a person slowly walked forward,a person is walking in a counterclockwise circle.,M002945,"a man walks forward, then turns around and wal..."
8,005799,a person walks forward and then walks back,a person slowly walked forward,a person sways to the side and stumbles forward.,012015,man walks forward while upper body is leaning ...
9,000749,a person stands still and then stands up,a person slowly walked forward,person is standing up moving objects.,002121,a man using both hands to lift something off g...


## Reusable outputs

Checkpoint, metrics, predictions, aur training history Google Drive ke `MyDrive/Pose2Text/` folder mein persist rahenge.


In [ ]:
from pathlib import Path
import zipfile
out_zip = PERSIST_ROOT / 'Pose2Text_outputs.zip'
with zipfile.ZipFile(out_zip, 'w', zipfile.ZIP_DEFLATED) as z:
    for folder in (RESULTS_DIR, CHECKPOINT_DIR):
        if folder.exists():
            for f in folder.rglob('*'):
                if f.is_file() and f.name != '.gitkeep': z.write(f, f.relative_to(PERSIST_ROOT))
print('Saved permanently in Drive:', out_zip)
print('You only need to train once. Next Colab session will reuse the checkpoint/results above.')
try:
    from google.colab import files
    files.download(str(out_zip))
except ImportError:
    pass


Saved permanently in Drive: /content/drive/MyDrive/Pose2Text/Pose2Text_outputs.zip
You only need to train once. Next Colab session will reuse the checkpoint/results above.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Encoder Self-Attention: Motion sequence ke different time steps ke relationships learn karta hai.



Decoder Masked Self-Attention: Future caption tokens ko hide karta hai, taaki model next word predict kar sake.



Cross-Attention: Motion ki information ko generated caption ke saath connect karta hai.

The encoder uses self-attention to learn relationships between motion time steps. The decoder uses masked self-attention to prevent access to future caption tokens and cross-attention to connect the encoded motion with caption generation